In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
project_dir_candidates = [
    Path.cwd(),
    Path.cwd() / "revision-fcgma-copy",
    Path.cwd().parent / "revision-fcgma-copy",
]
project_dir = next(
    (path for path in project_dir_candidates if (path / "maxCompNumber.ipynb").exists()),
    None,
)
if project_dir is None:
    raise FileNotFoundError("Could not locate the revision-fcgma-copy project directory.")

preprocessing_dir = next(
    (path for path in [project_dir.parent / "Preprocessing", project_dir / "Preprocessing"] if path.exists()),
    None,
)
if preprocessing_dir is None:
    raise FileNotFoundError("Could not locate the Preprocessing directory.")

size_path = preprocessing_dir / "preprocessed_final.csv"
output_path = project_dir / "max_comp_number.csv"
df_size = pd.read_csv(size_path, sep=';', decimal=',', encoding='utf-8-sig', engine = "python")

In [3]:
df_size.info()

<class 'pandas.DataFrame'>
RangeIndex: 10321 entries, 0 to 10320
Data columns (total 12 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   item_code  10321 non-null  int64  
 1   註記         10321 non-null  str    
 2   (箱)長cm     9646 non-null   float64
 3   (箱)寬cm     9646 non-null   float64
 4   (箱)高cm     9646 non-null   float64
 5   (箱)重量      9646 non-null   float64
 6   箱入數        10055 non-null  float64
 7   板入數        10055 non-null  str    
 8   庫存量        9966 non-null   float64
 9   庫存單位       10321 non-null  str    
 10  庫存換算(箱)    9718 non-null   float64
 11  目前庫存(PCS)  10130 non-null  float64
dtypes: float64(8), int64(1), str(3)
memory usage: 967.7 KB


In [4]:
# Filter the SKU to certain size

cols = ['carton_length_cm', 'carton_width_cm', 'carton_height_cm', 'units_per_carton']
df_size[cols] = df_size[cols].apply(pd.to_numeric, errors='coerce')
df_size = df_size.dropna(subset=cols).copy()

# The longest dimension
df_size['highest_dimension'] = df_size[['carton_length_cm', 'carton_width_cm', 'carton_height_cm']].max(axis=1)

# The second longest dimension
df_size['second_highest_dimension'] = df_size[['carton_length_cm', 'carton_width_cm', 'carton_height_cm']].apply(lambda x: sorted(x)[-2], axis=1)

# The shortest dimension
df_size['shortest_dimension'] = df_size[['carton_length_cm', 'carton_width_cm', 'carton_height_cm']].min(axis=1)

df_size = df_size[
    (df_size['highest_dimension'] <= 118) &
    (df_size['second_highest_dimension'] <= 48) &
    (df_size['shortest_dimension'] <= 45)
]

In [5]:
df_size.info()

<class 'pandas.DataFrame'>
Index: 9519 entries, 0 to 10319
Data columns (total 15 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   item_code                 9519 non-null   int64  
 1   註記                        9519 non-null   str    
 2   (箱)長cm                    9519 non-null   float64
 3   (箱)寬cm                    9519 non-null   float64
 4   (箱)高cm                    9519 non-null   float64
 5   (箱)重量                     9519 non-null   float64
 6   箱入數                       9519 non-null   float64
 7   板入數                       9519 non-null   str    
 8   庫存量                       9195 non-null   float64
 9   庫存單位                      9519 non-null   str    
 10  庫存換算(箱)                   9195 non-null   float64
 11  目前庫存(PCS)                 9338 non-null   float64
 12  highest_dimension         9519 non-null   float64
 13  second_highest_dimension  9519 non-null   float64
 14  shortest_dimension     

In [6]:
df_size = df_size[
    (df_size['carton_width_cm'] > 0) &
    (df_size['carton_height_cm'] > 0) &
    (df_size['carton_length_cm'] > 0) &
    (df_size['units_per_carton'] > 0)
].copy()

df_size["cartons_per_slot"] = np.floor(
    (118 / df_size['highest_dimension'])
    * (48 / df_size['second_highest_dimension'])
    * (45 / df_size['shortest_dimension'])
).astype(int)

df_size["max_fit"] = (
    df_size["cartons_per_slot"] * df_size['units_per_carton']
).astype(int)

In [7]:
df_size.head(20)

      item_code                 註記  ...  shortest_dimension  max_fit
0   10000001001     可口可樂Zero 600ml  ...                24.0        7
1   10000002001             可口可樂2L  ...                23.0       10
2   10000003001          百事可樂330ml  ...                12.0       19
3   10000004001             百事可樂2L  ...                23.0       10
4   10000005001          可口可樂330ml  ...                12.0       18
5   10000006001      Dr. Pepper 可樂  ...                13.0       37
6   10000007001    可口可樂 Zero 330ml  ...                12.0       18
7   10000008001     可口可樂纖維+ 1250ml  ...                26.0        9
8   10000010001       可口可樂 Zero 2L  ...                23.0       10
9   10000011001    可口可樂 Can 250 ml  ...                10.0       23
10  10000012001      可口可樂Pet 600ml  ...                25.0        7
11  10000013001      百事可樂 Can250ml  ...                10.0       22
12  10000021001  可口可樂纖維+ Can 330ml  ...                12.0       18
13  10000022001         可口可樂1250ml

In [8]:
# Save the result to a new CSV file
df_size = df_size.filter(items=['item_code', 'carton_length_cm', 'carton_width_cm', 'carton_height_cm', 'units_per_carton', 'cartons_per_slot', 'max_fit'])
try:
    df_size.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"Saved max-fit data to {output_path}")
except PermissionError:
    fallback_output_path = output_path.with_name(f"{output_path.stem}_generated{output_path.suffix}")
    df_size.to_csv(fallback_output_path, index=False, encoding="utf-8-sig")
    print(f"{output_path} is locked. Saved max-fit data to {fallback_output_path} instead.")

Saved max-fit data to D:\ITB\Tugas Akhir\revision-fcgma-copy\max_comp_number.csv
